# Практика 11. Числові характеристики випадкової величини; закон великих чисел і ЦГТ симуляцією

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса  
**Параметр експоненційного розподілу:** λ = 0.15 подій/хв

Теоретичне середнє часу очікування для експоненційного розподілу дорівнює `1/λ`. У цій практиці змоделюємо час очікування служби підтримки, перевіримо закон великих чисел і центральну граничну теорему (ЦГТ), а також порівняємо теоретичну стандартну похибку з результатами симуляції.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
lambda_value = 0.15
true_mean = 1 / lambda_value

print(f"Теоретичне середнє часу очікування: {true_mean:.4f} хв")

## Завдання 1. Перевірка закону великих чисел

Згенеруємо по одній вибірці для кожного `n`. Для кожного обсягу порівняємо вибіркове середнє з відомим теоретичним середнім `1/λ`.

In [ ]:
lln_results = []
for n in [5, 50, 500, 20_000]:
    sample = rng.exponential(scale=1 / lambda_value, size=n)
    sample_mean = sample.mean()
    lln_results.append({
        "n": n,
        "Вибіркове середнє, хв": sample_mean,
        "Абсолютна похибка, хв": abs(sample_mean - true_mean),
    })

lln_table = pd.DataFrame(lln_results)
display(lln_table)

Зі зростанням `n` середнє зазвичай наближається до теоретичного значення, а абсолютна похибка має тенденцію зменшуватися. Однак це не обов'язково відбувається рівномірно для кожної послідовної вибірки: таблиця показує конкретні випадкові результати, а закон великих чисел описує зближення зі зростанням обсягу вибірки.

## Завдання 2. Вибірковий розподіл середнього для двох n

Згенеруємо по 10 000 незалежних вибірок кожного розміру. Кожен елемент масиву `means_*` — середнє окремої вибірки, тому гістограми описують розподіл вибіркового середнього, а не окремих часів очікування.

In [ ]:
n_repeats = 10_000
means_n10 = rng.exponential(scale=1 / lambda_value, size=(n_repeats, 10)).mean(axis=1)
means_n300 = rng.exponential(scale=1 / lambda_value, size=(n_repeats, 300)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(means_n10, bins=40, density=True, edgecolor="white", alpha=0.8)
axes[0].axvline(true_mean, color="crimson", linestyle="--", label="Теоретичне середнє")
axes[0].set_title(f"Розподіл середнього, n=10 (std={means_n10.std():.3f})")
axes[0].set_xlabel("Середнє часу очікування, хв")
axes[0].set_ylabel("Густина")
axes[0].legend()

axes[1].hist(means_n300, bins=40, density=True, edgecolor="white", alpha=0.8)
axes[1].axvline(true_mean, color="crimson", linestyle="--", label="Теоретичне середнє")
axes[1].set_title(f"Розподіл середнього, n=300 (std={means_n300.std():.3f})")
axes[1].set_xlabel("Середнє часу очікування, хв")
axes[1].set_ylabel("Густина")
axes[1].legend()

plt.tight_layout()
plt.show()

За ЦГТ розподіл середнього для `n=10` ще може бути помітно скошеним вправо, тоді як для `n=300` він має бути значно симетричнішим і дзвоноподібним. Зі збільшенням розміру кожної вибірки також зменшується розкид її середнього навколо теоретичного середнього.

## Завдання 3. Стандартна похибка: теорія проти симуляції

Для експоненційного розподілу стандартне відхилення дорівнює `1/λ`, тому теоретична стандартна похибка середнього — `(1/λ) / √n`. Для кожного `n` порівняємо її зі стандартним відхиленням 10 000 змодельованих середніх.

In [ ]:
se_results = []
for n in [10, 30, 100, 300]:
    sample_means = rng.exponential(
        scale=1 / lambda_value,
        size=(n_repeats, n),
    ).mean(axis=1)
    theoretical_se = (1 / lambda_value) / np.sqrt(n)
    simulated_std = sample_means.std()
    se_results.append({
        "n": n,
        "Теоретична SE, хв": theoretical_se,
        "Симульована std середніх, хв": simulated_std,
        "Абсолютна різниця, хв": abs(theoretical_se - simulated_std),
    })

se_table = pd.DataFrame(se_results)
display(se_table)

У таблиці симульоване стандартне відхилення середніх має бути близьким до теоретичної SE. Вони не зобов'язані збігатися точно: симуляція використовує скінченну кількість повторів, а отже її оцінка також має випадкову похибку.

## Завдання 4. Закон убуваючої віддачі

Порахуємо теоретичну стандартну похибку для `n=25` і `n=100`. За формулою SE пропорційна `1/√n`.

In [ ]:
se_n25 = (1 / lambda_value) / np.sqrt(25)
se_n100 = (1 / lambda_value) / np.sqrt(100)

print(f"Теоретична SE для n=25: {se_n25:.4f} хв")
print(f"Теоретична SE для n=100: {se_n100:.4f} хв")
print(f"SE для n=100 становить {se_n100 / se_n25:.2f} від SE для n=25")

Збільшення `n` з 25 до 100 є чотириразовим, але SE зменшується вдвічі, а не вчетверо: у формулі стоїть квадратний корінь із `n`. Загалом, щоб зменшити SE вдвічі, потрібно збільшити кількість спостережень у чотири рази.

## Завдання 5. Вплив числа повторів на гістограму (додатково)

Тут розмір кожної вибірки залишається `n=10`, але кількість повторів зменшено з 10 000 до 50. Порівняйте гістограму з гістограмою `means_n10` із Завдання 2.

In [ ]:
n_repeats_small = 50
means_n10_few_repeats = rng.exponential(
    scale=1 / lambda_value,
    size=(n_repeats_small, 10),
).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(means_n10, bins=40, density=True, edgecolor="white", alpha=0.8)
axes[0].set_title("n=10, 10 000 повторів")
axes[0].set_xlabel("Середнє часу очікування, хв")
axes[0].set_ylabel("Густина")
axes[1].hist(means_n10_few_repeats, bins=15, density=True, edgecolor="white", alpha=0.8)
axes[1].set_title("n=10, 50 повторів")
axes[1].set_xlabel("Середнє часу очікування, хв")
axes[1].set_ylabel("Густина")
plt.tight_layout()
plt.show()

За 50 повторами гістограма зазвичай нерівна й має небагато стовпців із помітними випадковими коливаннями; 10 000 повторів дають набагато надійніше та плавніше уявлення про форму розподілу середнього. За гладкість гістограми відповідає кількість повторів `n_repeats`, бо вона визначає, скільки середніх спостережено. Розмір `n` натомість впливає на форму й розкид самого вибіркового розподілу середнього.

## Контрольні питання

1. **Чим відрізняються закон великих чисел і ЦГТ?** Закон великих чисел відповідає на питання, куди прямує вибіркове середнє `x̄`: зі зростанням обсягу вибірки воно наближається до теоретичного середнього. ЦГТ відповідає на питання про форму та розкид розподілу `x̄`: за достатньо великого `n` його розподіл стає приблизно нормальним, із центром біля теоретичного середнього.

2. **Навіщо вихідний розподіл сильно скошений?** Експоненційний розподіл часу очікування має праву скошеність, тож можна побачити, що середні багатьох вибірок набувають приблизно дзвоноподібної форми навіть тоді, коли початкові дані не є нормальними. Якби вихідні дані вже були нормальними, цей перехід був би менш наочним.

3. **Чому стандартна похибка зменшується як `1/√n`, а не як `1/n`?** За формулою `SE = σ/√n` зростання кількості незалежних спостережень зменшує невизначеність повільніше, ніж лінійно. Щоб зменшити SE вдвічі, треба збільшити `n` учетверо, бо `√(4n) = 2√n`.

## Підсумок

Для експоненційного часу очікування з λ = 0.15 подій/хв симуляція демонструє зближення вибіркового середнього з `1/λ`, перехід розподілу середнього до дзвоноподібної форми зі збільшенням `n` та узгодження симульованого розкиду середніх із формулою `SE = (1/λ)/√n`. Додаткове порівняння показує, що надійність гістограми залежить від числа повторних вибірок.